# HUE Real-Data Energy Distribution Optimization
This notebook discovers the project and HUE Kaggle inputs, validates the implementation, trains D3QN-PER and uniform replay, runs MPC and Rule-Based baselines, completes every requested sweep, audits the results, and creates a downloadable ZIP.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
PROJECT = WORK / 'energy_distribution_project'

def project_candidates(root):
    return [p.parent for p in root.rglob('run_experiment.py')
            if (p.parent / 'microgrid_opt' / 'data.py').exists()]

candidates = project_candidates(INPUT)
if not candidates:
    extracted = WORK / '_project_zip_extract'
    extracted.mkdir(exist_ok=True)
    for archive in INPUT.rglob('*.zip'):
        with zipfile.ZipFile(archive) as zf:
            names = zf.namelist()
            if any(n.endswith('run_experiment.py') for n in names) and any('microgrid_opt/data.py' in n for n in names):
                zf.extractall(extracted)
    candidates = project_candidates(extracted)
if not candidates:
    raise FileNotFoundError('Project input not found. Attach this project ZIP/folder to the notebook.')
source_project = sorted(candidates, key=lambda p: ('HUE_REALDATA' in str(p), len(str(p))), reverse=True)[0]
if PROJECT.exists():
    shutil.rmtree(PROJECT)
shutil.copytree(source_project, PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print('Project:', PROJECT)
print((PROJECT / 'VERSION.txt').read_text())

In [ ]:
from microgrid_opt.data import find_hue_root
HUE_ROOT = find_hue_root(INPUT)
print('HUE root:', HUE_ROOT)
print('Residential files:', len(list(HUE_ROOT.glob('Residential_*.csv'))))
assert (HUE_ROOT / 'Solar.csv').exists()

## Configuration
`full` is the client run. Use `preliminary` only when checking the pipeline or preparing a rapid progress update.

In [ ]:
RUN_MODE = 'full'                 # 'full' or 'preliminary'
N_PROSUMERS = 10
OUTAGE_SCENARIO = 'moderate'      # none, normal, moderate, stress
FULL_EPISODES = 30
FULL_TRAINING_WINDOW_DAYS = 90
PRELIMINARY_EPISODES = 5
PRELIMINARY_TRAINING_WINDOW_DAYS = 30
PRELIMINARY_TEST_HOURS = 1440
RESULTS = WORK / ('HUE_CLIENT_RESULTS' if RUN_MODE == 'full' else 'HUE_PRELIMINARY_RESULTS')
print('Mode:', RUN_MODE, '| Results:', RESULTS)

In [ ]:
# Install only if Kaggle is missing a dependency.
required = ['numpy', 'pandas', 'scipy', 'matplotlib']
missing = []
for package in required:
    try:
        __import__(package)
    except ImportError:
        missing.append(package)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
subprocess.run([sys.executable, 'scripts/run_self_checks.py'], check=True)

In [ ]:
# Preflight: print the actual selected houses and non-overlapping chronology before training.
from microgrid_opt.data import prepare_hue_community, chronological_split
preview, metadata = prepare_hue_community(HUE_ROOT, n_prosumers=N_PROSUMERS, outage_scenario=OUTAGE_SCENARIO)
train, validation, test, split = chronological_split(preview)
display(metadata['prosumer_mapping'])
display(split)
print('Test outage hours:', test.groupby('timestamp').grid_available.first().eq(0).sum())
del preview, train, validation, test

In [ ]:
cmd = [
    sys.executable, 'run_experiment.py',
    '--hue-root', str(HUE_ROOT),
    '--output', str(RESULTS),
    '--prosumers', str(N_PROSUMERS),
    '--outage-scenario', OUTAGE_SCENARIO,
    '--full',
]
if RUN_MODE == 'full':
    cmd += ['--episodes', str(FULL_EPISODES), '--training-window-days', str(FULL_TRAINING_WINDOW_DAYS)]
else:
    cmd += ['--episodes', str(PRELIMINARY_EPISODES),
            '--training-window-days', str(PRELIMINARY_TRAINING_WINDOW_DAYS),
            '--max-test-hours', str(PRELIMINARY_TEST_HOURS)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
audit = [sys.executable, 'scripts/audit_outputs.py', str(RESULTS)]
if RUN_MODE != 'full':
    audit.append('--allow-preliminary')
subprocess.run(audit, check=True)

import pandas as pd
display(pd.read_csv(RESULTS / 'RUN_MANIFEST.csv'))
display(pd.read_csv(RESULTS / 'tables' / 'd3qn_validation_summary.csv'))
display(pd.read_csv(RESULTS / 'tables' / 'switching_window_summary.csv'))
display(pd.read_csv(RESULTS / 'tables' / 'allocation_method_comparison.csv').head(24))

In [ ]:
archive_base = WORK / RESULTS.name
archive = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=RESULTS))
print('DOWNLOAD FROM KAGGLE OUTPUT:', archive)
print('Size (MB):', round(archive.stat().st_size / 1e6, 2))
print('Then use Save Version > Save & Run All to preserve it.')